In [3]:
%load_ext autoreload
%autoreload 2

from tgraasp import STx_encoder, STx_discriminator, STx_ARGVA
from tgraasp import DatasetScheme, TGraaspDataBuilder

sample_ids = ['ZWJBT','CAQSC']
raw_dir    = '/share/gguilin/szy/commercial/manman/model/data'
builder = TGraaspDataBuilder(raw_dir=raw_dir)
graph_scheme = DatasetScheme(
    counts_tmpl="{sid}_hvg_sp_data.txt",
    labels_tmpl="{sid}_meta_sp.txt",
    positions_tmpl="{sid}_position_sp.txt",
    label_col="integrated_snn_res.0.4",
    sep=r"\s+",
    with_edges=True,
    coord_cols=(1, 2), 
)
sp_graph_list, sp_adj_list = builder.build_dataset(
    sample_ids=sample_ids,
    scheme=graph_scheme,
    test_ratio=0.1,
    distance_ref_k=100,
    distance_ref_rank=8,
    distance_margin=0.0,
)

basic_scheme = DatasetScheme(
    counts_tmpl="{sid}_hvg_data.txt",
    labels_tmpl="{sid}_meta.txt",
    sep=r"\s+",
    with_edges=False,
)
sp_basic_list = builder.build_dataset(
    sample_ids=sample_ids,
    scheme=basic_scheme,
)

N_nodes   = sp_graph_list[0].x.size(0)   # 节点个数
F_inputs  = sp_graph_list[0].x.size(1)   

ppi_file     = f'{raw_dir}/PPI.connect1.txt'
connectivity = builder.load_ppi_connectivity(ppi_file, F_inputs)    # shape = [2, E]

encoder = STx_encoder(
    in_channels      = F_inputs,        
    hidden_channels  = 96,
    out_channels     = 16,
    m = 18, l = 3, K = 3,
    connect = connectivity,        
    pi = 0.75,
    n_heads = 8,
)

discriminator = STx_discriminator(
    in_channels     = 16,  
    hidden_channels = 64,   
    out_channels    = 1
)

model = STx_ARGVA(
    encoder      = encoder,
    discriminator= discriminator,
    l            = 3          
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[build_dataset] Loading sample ZWJBT (with_edges=True)
[build_dataset] Loading sample CAQSC (with_edges=True)
[build_dataset] Loading sample ZWJBT (with_edges=False)
[build_dataset] Loading sample CAQSC (with_edges=False)


In [4]:
from tgraasp.train import TGraaspTrainer
trainer = TGraaspTrainer(
    model=model,
    sp_graphs=sp_graph_list,
    sc_graphs=sp_basic_list,
    sp_adjs=sp_adj_list,
    device='cuda',
    pretrain_epochs=500)
best_model_state = trainer.pretrain()

=== TGraaspTrainer: Data Debug ===
Spatial datasets: 2
scRNA datasets:  2
Spatial adjs:    2
Start Pretrain...
[Pretrain] Epoch 001 | Dataset 00 | ARI 0.063 | ACC 0.173 | NMI 0.135 | AUC 0.511 | AP 0.556 | ACC1 0.500
[Pretrain] Epoch 001 | Dataset 01 | ARI 0.156 | ACC 0.197 | NMI 0.202 | AUC 0.480 | AP 0.535 | ACC1 0.500
✔ New best mean AUROC 0.4954 at epoch 1
Epoch 001 | Avg Loss 19.0106 | Mean AUC 0.4954 | Best 0.4954 (Epoch 1)
[Pretrain] Epoch 002 | Dataset 00 | ARI 0.115 | ACC 0.242 | NMI 0.178 | AUC 0.510 | AP 0.564 | ACC1 0.500
[Pretrain] Epoch 002 | Dataset 01 | ARI 0.257 | ACC 0.299 | NMI 0.277 | AUC 0.487 | AP 0.544 | ACC1 0.500
✔ New best mean AUROC 0.4989 at epoch 2
Epoch 002 | Avg Loss 18.9120 | Mean AUC 0.4989 | Best 0.4989 (Epoch 2)
[Pretrain] Epoch 003 | Dataset 00 | ARI 0.174 | ACC 0.308 | NMI 0.230 | AUC 0.512 | AP 0.568 | ACC1 0.500
[Pretrain] Epoch 003 | Dataset 01 | ARI 0.351 | ACC 0.381 | NMI 0.353 | AUC 0.492 | AP 0.547 | ACC1 0.500
✔ New best mean AUROC 0.5019 at

In [5]:
from tgraasp.models import Alignment_Model

trainer.load_best_model()
finetuned_best_model_state = trainer.finetune_alignment(
    F_inputs=F_inputs,
    Alignment_Model=Alignment_Model,
    epochs=100,
    patience=20,
    lr=1e-4,
)

/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


[Finetune] Epoch 001 | Avg Loss: 0.689987 | Avg Align Loss (Train): 0.689987 | Avg Align Loss (Eval): 0.354154 | Avg Corr: nan | Avg AUC: 0.8345


/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


[Finetune] Epoch 005 | Avg Loss: 0.450658 | Avg Align Loss (Train): 0.323166 | Avg Align Loss (Eval): 0.284852 | Avg Corr: nan | Avg AUC: 0.8353


/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


[Finetune] Epoch 010 | Avg Loss: 0.286301 | Avg Align Loss (Train): 0.274702 | Avg Align Loss (Eval): 0.250753 | Avg Corr: nan | Avg AUC: 0.8360


/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


[Finetune] Epoch 015 | Avg Loss: 0.239956 | Avg Align Loss (Train): 0.217108 | Avg Align Loss (Eval): 0.203348 | Avg Corr: nan | Avg AUC: 0.8342


/share/gguilin/miniconda3/envs/GCN/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4424: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


[Finetune] Epoch 020 | Avg Loss: 0.188031 | Avg Align Loss (Train): 0.169931 | Avg Align Loss (Eval): 0.165468 | Avg Corr: nan | Avg AUC: 0.8350
[Finetune] Epoch 025 | Avg Loss: 0.157393 | Avg Align Loss (Train): 0.150691 | Avg Align Loss (Eval): 0.143795 | Avg Corr: 0.0179 | Avg AUC: 0.8351
✔ New best (corr=0.0179) at epoch 25
[Finetune] Epoch 030 | Avg Loss: 0.146257 | Avg Align Loss (Train): 0.144166 | Avg Align Loss (Eval): 0.132397 | Avg Corr: -0.0122 | Avg AUC: 0.8350
[Finetune] Epoch 035 | Avg Loss: 0.133507 | Avg Align Loss (Train): 0.125916 | Avg Align Loss (Eval): 0.102995 | Avg Corr: 0.0143 | Avg AUC: 0.8354
[Finetune] Epoch 040 | Avg Loss: 0.112462 | Avg Align Loss (Train): 0.104822 | Avg Align Loss (Eval): 0.081797 | Avg Corr: 0.1753 | Avg AUC: 0.8353
✔ New best (corr=0.1753) at epoch 40
[Finetune] Epoch 045 | Avg Loss: 0.096379 | Avg Align Loss (Train): 0.090720 | Avg Align Loss (Eval): 0.067656 | Avg Corr: 0.2744 | Avg AUC: 0.8352
✔ New best (corr=0.2744) at epoch 45
[Fi